# Synergy Evaluation: v1 vs v2 Non-Agentic vs v2 Agentic

Replicates the Synergy dataset evaluation using v2. Same 3-way comparison as custom evaluation.
Models: `gemini-3.1-flash-lite-preview` + `gpt-5.4-mini`

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')
import sys
sys.path.append('../')

In [ ]:
import glob, json, shutil, warnings, pickle
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, roc_curve, f1_score)
from pydantic import BaseModel, Field
from lattereview.agentic import AgenticReviewer, AgenticWorkflow

## Infrastructure (models, prompts, workflow)

In [ ]:
class TitleAbstractOutput(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning for the evaluation.")
    evaluation: int = Field(ge=1, le=5, description="Evaluation score: 1=absolutely exclude, 2=better to exclude, 3=not sure, 4=better to include, 5=absolutely include.")
    certainty: int = Field(ge=0, le=100, description="Confidence in the evaluation (0-100).")

SYSTEM_PROMPT = (
    "You are an expert systematic reviewer. Your task is to evaluate whether "
    "studies should be included or excluded based on their title and abstract. "
    "Be thorough and apply the criteria consistently."
)
TASK_PROMPT_TEMPLATE = (
    "**Review the title and abstract below and evaluate whether they should be "
    "included based on the following inclusion and exclusion criteria (if any).**\n"
    "**Note that the study should be included only and only if it meets ALL "
    "inclusion criteria and NONE of the exclusion criteria.**\n\n---\n\n"
    "**Input item:**\n<<${item}$>>\n\n---\n\n"
    "**Inclusion criteria:**\nINCLUSION_CRITERIA_PLACEHOLDER\n\n"
    "**Exclusion criteria:**\nEXCLUSION_CRITERIA_PLACEHOLDER\n\n---\n\n"
    "**Instructions**\n\n"
    "1. Output your evaluation as an integer between 1 and 5, where:\n"
    "   - 1 means absolutely to exclude.\n   - 2 means better to exclude.\n"
    "   - 3 Not sure if to include or exclude.\n   - 4 means better to include.\n"
    "   - 5 means absolutely to include.\n"
    "2. Report your certainty level between **0** and **100**.\n"
    "3. Provide your reasoning before assigning a decision."
)
SENIOR_SYSTEM_PROMPT = (
    "You are a senior expert systematic reviewer. Two junior reviewers have already "
    "reviewed this article. Use your expertise to make the final determination."
)

def create_reviewers(inclusion_criteria, exclusion_criteria, agentic=False):
    task_prompt = TASK_PROMPT_TEMPLATE.replace("INCLUSION_CRITERIA_PLACEHOLDER", str(inclusion_criteria)).replace("EXCLUSION_CRITERIA_PLACEHOLDER", str(exclusion_criteria))
    ca = dict(agentic_effort="high", skills=["searching-duckduckgo"])
    gak = {**ca, "max_iterations": 30} if agentic else {"max_iterations": 1}
    oak = {**ca, "max_iterations": 15} if agentic else {"max_iterations": 1}
    A1 = AgenticReviewer(name="Agent1", model="google-gla:gemini-3.1-flash-lite-preview", backstory="a PhD researcher", system_prompt=SYSTEM_PROMPT, task_prompt=task_prompt, output_type=TitleAbstractOutput, max_concurrent_requests=20 if not agentic else 10, model_settings={"temperature": 0.1}, **gak)
    A2 = AgenticReviewer(name="Agent2", model="openai:gpt-5.4-mini", backstory="a PhD researcher", system_prompt=SYSTEM_PROMPT, task_prompt=task_prompt, output_type=TitleAbstractOutput, max_concurrent_requests=20 if not agentic else 10, model_settings={"temperature": 0.1}, **oak)
    A3 = AgenticReviewer(name="Agent3", model="openai:gpt-5.4-mini", backstory="a senior MD-PhD researcher", system_prompt=SENIOR_SYSTEM_PROMPT, task_prompt=task_prompt, output_type=TitleAbstractOutput, max_concurrent_requests=20 if not agentic else 10, model_settings={"temperature": 0.1}, **oak)
    return A1, A2, A3

def split_dataframe(df, n):
    return [df.iloc[i:i+n] for i in range(0, len(df), n)]

async def run_evaluation(name, inc, exc, df, agentic=False, wdir=None):
    mode = "agentic" if agentic else "non_agentic"
    print(f"\n{'='*60}\n{name} — {mode}\n{'='*60}")
    chunks = split_dataframe(df, 1000)
    results, cost = [], 0.0
    for ci, sub in enumerate(chunks):
        A1, A2, A3 = create_reviewers(inc, exc, agentic)
        def ff(row):
            try:
                s1, s2 = int(row["round-A_Agent1_output"]["evaluation"]), int(row["round-A_Agent2_output"]["evaluation"])
            except (TypeError, KeyError, ValueError):
                return False
            if s1 != s2:
                if s1 >= 4 and s2 >= 4: return False
                if s1 >= 3 or s2 >= 3: return True
            elif s1 == s2 == 3: return True
            return False
        wd = Path(wdir)/f"{name}_{mode}_chunk{ci}" if wdir and agentic else None
        if wd and wd.exists(): shutil.rmtree(wd)
        wf = AgenticWorkflow(workflow_schema=[
            {"round": "A", "reviewers": [A1, A2], "text_inputs": ["title", "abstract"]},
            {"round": "B", "reviewers": [A3], "text_inputs": ["title", "abstract", "round-A_Agent1_output", "round-A_Agent2_output"], "filter": ff},
        ], working_dir=wd, verbose=True)
        r = await wf(sub); results.append(r); cost += wf.total_cost
    return pd.concat(results), cost

def get_score(row):
    if "round-B_Agent3_output" in row and pd.notna(row.get("round-B_Agent3_evaluation")):
        try: return int(row["round-B_Agent3_evaluation"])
        except (TypeError, ValueError): pass
    try: return (int(row["round-A_Agent1_evaluation"]) + int(row["round-A_Agent2_evaluation"])) / 2
    except (TypeError, ValueError): return 1

def evaluate_metrics(gt, preds, ts=1.5, tb=3.0, tp=4.5):
    cl = lambda p, t: [1 if x >= t else 0 for x in p]
    m = {}
    for l, t in [('sensitive', ts), ('balanced', tb), ('specific', tp)]:
        p = cl(preds, t); tn, fp, fn, tp_ = confusion_matrix(gt, p, labels=[0,1]).ravel()
        m[f'accuracy_{l}'] = accuracy_score(gt, p); m[f'precision_{l}'] = precision_score(gt, p, zero_division=0)
        m[f'recall_{l}'] = recall_score(gt, p, zero_division=0); m[f'f1_{l}'] = f1_score(gt, p, zero_division=0)
        m[f'specificity_{l}'] = tn/(tn+fp) if (tn+fp)>0 else 0
    m['auc'] = roc_auc_score(gt, preds) if len(set(gt))>1 else float('nan')
    return m

def pm(name, gt, sc):
    m = evaluate_metrics(gt, sc)
    print(f"{name:<25} AUC={m['auc']:.3f}  F1(bal)={m['f1_balanced']:.3f}  Rec(bal)={m['recall_balanced']:.3f}  Prec(bal)={m['precision_balanced']:.3f}")
    return m

print("Infrastructure ready. Models: gemini-3.1-flash-lite + gpt-5.4-mini")

## Data & Criteria

In [ ]:
with open('synergy_data/all_review_jobs.pickle', 'rb') as f:
    all_review_jobs = pickle.load(f)
for name, df in all_review_jobs:
    print(f"{name}: {len(df)} items, {df['label_included'].sum()} positive")

criteria = {
    "appenzeller-herzog_2019": {"inc": "-Patients with Wilson's Disease of any age or stage\n-Study drug has to be one of four established therapies, namely DPen, trientine, TTM or Zn.\n-Control could be placebo, no treatment or any other treatment\n-Prospective or retrospective studies\n-Randomized, non-randomized controlled trials and comparative observational studies", "exc": "-Animal studies, case reports, case series, cross-sectional studies, before-after studies, reviews, letters, abstract-only publications, editorials, diagnostic or other testing studies and non-controlled studies"},
    "donners_2021": {"inc": "Emicizumab studies providing (1) data on humans, (2) original PK data or modeled PK data or PK/PD relationships, and (3) access to the abstract and full text in English.", "exc": "Not specified"},
    "jeyaraman_2020": {"inc": "Population: Patients with knee osteoarthritis. Intervention: MSC therapy. Comparator: Usual care. Outcomes: VAS, WOMAC, Lysholm, WORMS, KOOS, adverse events. Study Design: RCTs", "exc": "Observational studies without comparator group, animal studies, reviews"},
    "meijboom_2021": {"inc": "Studies involving transitioning from TNF-alpha inhibitor originator to biosimilar, with retransition data, original research, baseline characteristics, English.", "exc": "Not specified"},
    "muthu_2021": {"inc": "RCT with 1:1 parallel two-arm design, related to spine surgery, with dichotomous primary or secondary outcome.", "exc": "Non-human studies, continuous variable outcomes without clinical success criteria, studies without statistically significant outcomes"},
    "oud_2018": {"inc": "RCTs on DBT, MBT, TFP or ST for adults with BPD, including individual psychotherapy, 16+ weeks duration.", "exc": "Studies with <66% BPD participants, incomplete versions of specialized treatment"},
}

## Load v1 Baseline

In [ ]:
v1_raw = {}
for f in sorted(glob.glob("synergy_data/*_reviewed.csv")):
    df_v1 = pd.read_csv(f)
    name = f.split("/")[-1].replace("_reviewed.csv", "")
    gt = df_v1["label_included"].apply(int).tolist()
    def gvs(row):
        if "round-B_Agent3_output" in row and pd.notna(row.get("round-B_Agent3_evaluation")):
            return int(row["round-B_Agent3_evaluation"])
        return (int(row["round-A_Agent1_evaluation"]) + int(row["round-A_Agent2_evaluation"])) / 2
    sc = df_v1.apply(gvs, axis=1).tolist()
    v1_raw[name] = (gt, sc)
    print(f"v1 {name}: {len(gt)} items")

na_results = {}
ag_results = {}

## Donners_2021

In [ ]:
rn = "Donners_2021"
df = [d for n, d in all_review_jobs if n == rn][0]
c = criteria["donners_2021"]

r_na, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=False)
r_na.to_csv(f"synergy_data/{rn}_v2_nonagentic.csv", index=False)
na_results[rn] = r_na

r_ag, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=True, wdir="synergy_data/agentic_runs")
r_ag.to_csv(f"synergy_data/{rn}_v2_agentic.csv", index=False)
ag_results[rn] = r_ag

print("\n" + "="*60 + f"\n{rn} RESULTS\n" + "="*60)
v1g, v1s = v1_raw[rn]
gt = df["label_included"].apply(int).tolist()
pm("v1 (old models)", v1g, v1s)
pm("v2 Non-Agentic", gt, r_na.apply(get_score, axis=1).tolist())
pm("v2 Agentic", gt, r_ag.apply(get_score, axis=1).tolist())

## Meijboom_2021

In [ ]:
rn = "Meijboom_2021"
df = [d for n, d in all_review_jobs if n == rn][0]
c = criteria["meijboom_2021"]

r_na, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=False)
r_na.to_csv(f"synergy_data/{rn}_v2_nonagentic.csv", index=False)
na_results[rn] = r_na

r_ag, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=True, wdir="synergy_data/agentic_runs")
r_ag.to_csv(f"synergy_data/{rn}_v2_agentic.csv", index=False)
ag_results[rn] = r_ag

print("\n" + "="*60 + f"\n{rn} RESULTS\n" + "="*60)
v1g, v1s = v1_raw[rn]
gt = df["label_included"].apply(int).tolist()
pm("v1 (old models)", v1g, v1s)
pm("v2 Non-Agentic", gt, r_na.apply(get_score, axis=1).tolist())
pm("v2 Agentic", gt, r_ag.apply(get_score, axis=1).tolist())

## Oud_2018

In [ ]:
rn = "Oud_2018"
df = [d for n, d in all_review_jobs if n == rn][0]
c = criteria["oud_2018"]

r_na, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=False)
r_na.to_csv(f"synergy_data/{rn}_v2_nonagentic.csv", index=False)
na_results[rn] = r_na

r_ag, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=True, wdir="synergy_data/agentic_runs")
r_ag.to_csv(f"synergy_data/{rn}_v2_agentic.csv", index=False)
ag_results[rn] = r_ag

print("\n" + "="*60 + f"\n{rn} RESULTS\n" + "="*60)
v1g, v1s = v1_raw[rn]
gt = df["label_included"].apply(int).tolist()
pm("v1 (old models)", v1g, v1s)
pm("v2 Non-Agentic", gt, r_na.apply(get_score, axis=1).tolist())
pm("v2 Agentic", gt, r_ag.apply(get_score, axis=1).tolist())

## Jeyaraman_2020

In [ ]:
rn = "Jeyaraman_2020"
df = [d for n, d in all_review_jobs if n == rn][0]
c = criteria["jeyaraman_2020"]

r_na, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=False)
r_na.to_csv(f"synergy_data/{rn}_v2_nonagentic.csv", index=False)
na_results[rn] = r_na

r_ag, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=True, wdir="synergy_data/agentic_runs")
r_ag.to_csv(f"synergy_data/{rn}_v2_agentic.csv", index=False)
ag_results[rn] = r_ag

print("\n" + "="*60 + f"\n{rn} RESULTS\n" + "="*60)
v1g, v1s = v1_raw[rn]
gt = df["label_included"].apply(int).tolist()
pm("v1 (old models)", v1g, v1s)
pm("v2 Non-Agentic", gt, r_na.apply(get_score, axis=1).tolist())
pm("v2 Agentic", gt, r_ag.apply(get_score, axis=1).tolist())

## Muthu_2021

In [ ]:
rn = "Muthu_2021"
df = [d for n, d in all_review_jobs if n == rn][0]
c = criteria["muthu_2021"]

r_na, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=False)
r_na.to_csv(f"synergy_data/{rn}_v2_nonagentic.csv", index=False)
na_results[rn] = r_na

r_ag, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=True, wdir="synergy_data/agentic_runs")
r_ag.to_csv(f"synergy_data/{rn}_v2_agentic.csv", index=False)
ag_results[rn] = r_ag

print("\n" + "="*60 + f"\n{rn} RESULTS\n" + "="*60)
v1g, v1s = v1_raw[rn]
gt = df["label_included"].apply(int).tolist()
pm("v1 (old models)", v1g, v1s)
pm("v2 Non-Agentic", gt, r_na.apply(get_score, axis=1).tolist())
pm("v2 Agentic", gt, r_ag.apply(get_score, axis=1).tolist())

## Appenzeller-Herzog_2019

In [ ]:
rn = "Appenzeller-Herzog_2019"
df = [d for n, d in all_review_jobs if n == rn][0]
c = criteria["appenzeller-herzog_2019"]

r_na, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=False)
r_na.to_csv(f"synergy_data/{rn}_v2_nonagentic.csv", index=False)
na_results[rn] = r_na

r_ag, _ = await run_evaluation(rn, c["inc"], c["exc"], df.copy(), agentic=True, wdir="synergy_data/agentic_runs")
r_ag.to_csv(f"synergy_data/{rn}_v2_agentic.csv", index=False)
ag_results[rn] = r_ag

print("\n" + "="*60 + f"\n{rn} RESULTS\n" + "="*60)
v1g, v1s = v1_raw[rn]
gt = df["label_included"].apply(int).tolist()
pm("v1 (old models)", v1g, v1s)
pm("v2 Non-Agentic", gt, r_na.apply(get_score, axis=1).tolist())
pm("v2 Agentic", gt, r_ag.apply(get_score, axis=1).tolist())

## Final Comparison Plots

In [ ]:
all_m = {}
all_m["v1 (old models)"] = {n: {**evaluate_metrics(g, s), '_raw': (g, s)} for n, (g, s) in v1_raw.items()}
all_m["v2 Non-Agentic"] = {}
all_m["v2 Agentic"] = {}
for rn, df_na in na_results.items():
    df_orig = [d for n, d in all_review_jobs if n == rn][0]
    gt = df_orig["label_included"].apply(int).tolist()
    sc = df_na.apply(get_score, axis=1).tolist()
    all_m["v2 Non-Agentic"][rn] = {**evaluate_metrics(gt, sc), '_raw': (gt, sc)}
for rn, df_ag in ag_results.items():
    df_orig = [d for n, d in all_review_jobs if n == rn][0]
    gt = df_orig["label_included"].apply(int).tolist()
    sc = df_ag.apply(get_score, axis=1).tolist()
    all_m["v2 Agentic"][rn] = {**evaluate_metrics(gt, sc), '_raw': (gt, sc)}

conds = list(all_m.keys()); dsets = list(all_m[conds[0]].keys())
colors_c = ['#4C72B0', '#DD8452', '#55A868']; x = np.arange(len(dsets)); nc = len(conds); w = 0.8/nc

fig, ax = plt.subplots(figsize=(14, 5))
for i, c in enumerate(conds):
    v = [all_m[c][d]['auc'] for d in dsets]
    bars = ax.bar(x+i*w-(nc-1)*w/2, v, w, label=c, color=colors_c[i], alpha=0.85)
    for b, val in zip(bars, v):
        if not np.isnan(val): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{val:.2f}', ha='center', fontsize=7)
ax.set_ylabel('AUC'); ax.set_title('Synergy: AUC Comparison'); ax.set_xticks(x)
ax.set_xticklabels(dsets, rotation=30, ha='right'); ax.legend(); ax.set_ylim(0,1.1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
for i, c in enumerate(conds):
    v = [all_m[c][d]['f1_balanced'] for d in dsets]
    bars = ax.bar(x+i*w-(nc-1)*w/2, v, w, label=c, color=colors_c[i], alpha=0.85)
    for b, val in zip(bars, v): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{val:.2f}', ha='center', fontsize=7)
ax.set_ylabel('F1'); ax.set_title('Synergy: F1 (Balanced)'); ax.set_xticks(x)
ax.set_xticklabels(dsets, rotation=30, ha='right'); ax.legend(); ax.set_ylim(0,1.1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Summary table
rows = []
for c in conds:
    for d in dsets:
        m = all_m[c][d]
        rows.append({'Condition': c, 'Dataset': d, 'AUC': m['auc'], 'F1(bal)': m['f1_balanced'],
                     'Recall(bal)': m['recall_balanced'], 'Prec(bal)': m['precision_balanced'],
                     'F1(sens)': m['f1_sensitive'], 'Recall(sens)': m['recall_sensitive']})
pd.DataFrame(rows).round(3)